# 9. REPORTING DASHBOARD
## Daily Customer Churn Predictor · VivaMarket Brasil

---

**INPUT:** `../data/processed/churn_predictions_YYYYMMDD.parquet`, `../data/processed/churn_diagnostics_YYYYMMDD.csv`, `../data/processed/churn_explainability_YYYYMMDD.parquet`, and `../data/processed/retention_actions_YYYYMMDD.parquet`

**OUTPUT:** `../reports/churn_monitoring_dashboard_YYYYMMDD.html`

*A compact monitoring dashboard summarising risk mix, diagnostics, driver patterns, and retention action readiness — aligned with the canonical V2C policy and the operational n8n V9 workflow.*


---
## 9.1. STARTING SITUATION


NB08 has produced the full set of processed operational outputs: risk-scored customers (`churn_predictions`), explainability and driver assignments (`churn_explainability`), retention action payloads (`retention_actions`), and the diagnostic table (`churn_diagnostics`) generated by NB05.

What is still missing at this point is a **consolidated monitoring view** that brings all those outputs together for stakeholders. Each artefact lives in a separate file; there is no single place where a business user can check risk distribution, dominant churn drivers, campaign readiness, and model quality at a glance.

This notebook closes that gap by assembling all processed outputs into a single lightweight HTML dashboard aligned with the canonical V2C policy.

**Important tiering note — analytical vs. operational criteria.**
The risk tiers in this dashboard (`LOW / MEDIUM / HIGH`) are derived from **V2C percentile thresholds** (bottom 50% / next 30% / top 20% by score rank), which is the analytical definition used throughout NB05–NB08 for diagnostics, explainability, and action-payload generation.
The operational n8n V9 workflow routes customers using **fixed probability bands** (HIGH ≥ 0.75 / MEDIUM 0.45–0.75) applied at execution time, because at 02:00 the workflow has access to the raw `churn_probability` column rather than pre-computed tier labels.
Both approaches share the same tier names and ultimately target the same high-risk population, but the boundary conditions differ by design: percentile tiers are stable across score-distribution shifts (useful for trend analysis), while fixed bands are transparent and auditable at the workflow level. This distinction is documented here so any stakeholder reading the dashboard alongside the n8n workflow JSON understands why the exact customer counts at tier boundaries may diverge slightly.


---
## 9.2. NOTEBOOK OBJECTIVE


- **Business objective:** give stakeholders a single monitoring view for predictive risk, campaign readiness, and explainability.
- **Analytical objective:** consolidate diagnostics, driver mix, and action summaries into a lightweight HTML dashboard aligned with the canonical V2C policy.
- **Operational coherence objective:** surface the tiering-criterion divergence between the analytical pipeline (percentile-based V2C tiers) and the n8n V9 operational workflow (fixed probability bands: HIGH ≥ 0.75 / MEDIUM 0.45–0.75) explicitly in the dashboard Notes section, ensuring the artefact is fully auditable.

**What is done**

We import the standard libraries needed for data loading, date handling, and logging.

**Why it is done**

Centralising imports in the first cell makes dependencies explicit and ensures the notebook fails fast if a required package is missing rather than mid-execution.

**Expected result**

No output. A single INFO log line confirms the notebook has started.

In [1]:
import json
import logging
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import joblib
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', force=True)
logger = logging.getLogger('nb09_reporting_dashboard')
logger.info('NB09 started: reporting dashboard.')


2026-05-19 16:09:02,281 | INFO | NB09 started: reporting dashboard.


**What is done**

We resolve project paths, set the run date tag, and load the four processed artefacts produced by the upstream notebooks: predictions, explainability, retention actions, and diagnostics.

**Why it is done**

The dashboard must always reflect the most recent run. Glob-sorted path resolution guarantees that the latest file is picked without hardcoding dates.

**Expected result**

Four DataFrames in memory and a `dashboard_path` pointing to the target HTML file. No errors if the upstream notebooks have been executed successfully.

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

run_timestamp = datetime.now(ZoneInfo('Europe/Paris'))
run_date_tag = run_timestamp.strftime('%Y%m%d')
run_datetime_label = run_timestamp.strftime('%Y-%m-%d %H:%M %Z')
prediction_df = pd.read_parquet(sorted(PROCESSED_DIR.glob('churn_predictions_*.parquet'))[-1])
explainability_df = pd.read_parquet(sorted(PROCESSED_DIR.glob('churn_explainability_*.parquet'))[-1])
actions_df = pd.read_parquet(sorted(PROCESSED_DIR.glob('retention_actions_*.parquet'))[-1])
diagnostics_df = pd.read_csv(sorted(PROCESSED_DIR.glob('churn_diagnostics_*.csv'))[-1])
scoring_bundle = joblib.load(sorted(MODELS_DIR.glob('churn_scoring_package_*.joblib'))[-1])
model_version = scoring_bundle['metadata'].get('model_version', scoring_bundle['metadata'].get('version_name', 'v2'))
pipeline_tag = scoring_bundle['metadata'].get('pipeline_tag', 'canonical_v2c_phase2')
dashboard_path = REPORTS_DIR / f'churn_monitoring_dashboard_{run_date_tag}.html'


**What is done**

We compute five summary aggregations — risk mix (V2C percentile tiers), a side-by-side operational tier distribution using the n8n V9 fixed probability bands (HIGH ≥ 0.75 / MEDIUM 0.45–0.75), driver mix, action mix, and model diagnostics — and assemble them into a single HTML document with a notes section that explicitly documents the tiering-criterion divergence.

**Why it is done**

A single HTML file is the lightest possible shareable artefact: it requires no server, no dependencies, and no authentication. Tables are built from the canonical V2C aggregations so the content stays fully reproducible from the processed parquets.

Including the operational tier distribution table (derived from the same `churn_probability` column using the workflow's fixed bands) lets any reader cross-check the two classification criteria and understand why tier counts may diverge. This is the minimum necessary documentation to make the system fully auditable without requiring access to the n8n workflow JSON.

**Expected result**

An HTML file written to `../reports/churn_monitoring_dashboard_YYYYMMDD.html`. The logger confirms the save path. The cell returns the `PosixPath` of the file.

In [3]:
# --- Aggregation 1: Risk mix (V2C percentile tiers — analytical definition) ---
risk_mix = (
    prediction_df.groupby('risk_tier', observed=False)
    .agg(
        rows_n=('customer_unique_id', 'size'),
        avg_probability=('churn_probability', 'mean'),
        observed_churn=('observed_target', 'mean'),
    )
    .reset_index()
)
risk_mix['risk_tier'] = pd.Categorical(risk_mix['risk_tier'], categories=['HIGH', 'MEDIUM', 'LOW'], ordered=True)
risk_mix = risk_mix.sort_values('risk_tier')

# --- Aggregation 2: Operational tier distribution (n8n V9 fixed probability bands) ---
N8N_HIGH_THRESHOLD = 0.75
N8N_MEDIUM_THRESHOLD = 0.45

def assign_n8n_tier(prob):
    if prob >= N8N_HIGH_THRESHOLD:
        return 'HIGH'
    if prob >= N8N_MEDIUM_THRESHOLD:
        return 'MEDIUM'
    return 'LOW'

prediction_df['n8n_risk_tier'] = prediction_df['churn_probability'].apply(assign_n8n_tier)

operational_tier_dist = (
    prediction_df.groupby('n8n_risk_tier')
    .agg(
        rows_n=('customer_unique_id', 'size'),
        avg_probability=('churn_probability', 'mean'),
        observed_churn=('observed_target', 'mean'),
    )
    .reset_index()
    .rename(columns={'n8n_risk_tier': 'n8n_operational_tier'})
)
operational_tier_dist['n8n_operational_tier'] = pd.Categorical(operational_tier_dist['n8n_operational_tier'], categories=['HIGH', 'MEDIUM', 'LOW'], ordered=True)
operational_tier_dist = operational_tier_dist.sort_values('n8n_operational_tier')

# --- Aggregation 3: Top driver mix (V2C tiers) ---
driver_mix = (
    explainability_df.groupby(['risk_tier', 'top_driver_group'], observed=False)
    .size()
    .reset_index(name='rows_n')
)
driver_mix['share_within_tier'] = driver_mix.groupby('risk_tier', observed=False)['rows_n'].transform(lambda s: s / s.sum())

# --- Aggregation 4: Retention action mix (V2C tiers) ---
action_mix = (
    actions_df.groupby(['risk_tier', 'recommended_offer_type', 'primary_channels'], observed=False)
    .agg(
        rows_n=('customer_unique_id', 'size'),
        send_rows=('send_action_flag', 'sum'),
        control_rows=('control_group_flag', 'sum'),
        avg_discount_pct=('recommended_discount_pct', 'mean'),
    )
    .reset_index()
)

# --- Aggregation 5: Model diagnostics ---
summary_metrics = (
    diagnostics_df[diagnostics_df['section'] == 'summary'].copy()
    if 'section' in diagnostics_df.columns
    else diagnostics_df.copy()
)
summary_metrics = summary_metrics[['metric', 'value']].dropna(how='all')
threshold_metrics = (
    diagnostics_df[diagnostics_df['section'] == 'thresholds'].copy()
    if 'section' in diagnostics_df.columns
    else pd.DataFrame()
)
threshold_keep_cols = ['score_quantile_cutoff', 'score_threshold', 'targeted_rows', 'targeted_share', 'observed_churn_rate', 'recall_at_threshold', 'avg_total_payment_value']
threshold_metrics = threshold_metrics[[c for c in threshold_keep_cols if c in threshold_metrics.columns]].dropna(how='all')

precision_top10 = float(summary_metrics.loc[summary_metrics['metric'].eq('precision_at_top_10pct'), 'value'].iloc[0])
high_risk_rate = float(risk_mix.loc[risk_mix['risk_tier'].eq('HIGH'), 'observed_churn'].iloc[0])
roc_auc_value = float(summary_metrics.loc[summary_metrics['metric'].eq('roc_auc'), 'value'].iloc[0])
avg_precision_value = float(summary_metrics.loc[summary_metrics['metric'].eq('average_precision'), 'value'].iloc[0])

risk_donut = px.pie(
    risk_mix,
    names='risk_tier',
    values='rows_n',
    hole=0.55,
    color='risk_tier',
    color_discrete_map={'HIGH': '#c0392b', 'MEDIUM': '#d4a017', 'LOW': '#2e8b57'},
    title='Analytical Risk Mix (V2C Percentile Tiers)'
)
risk_donut.update_traces(textposition='inside', textinfo='percent+label')
risk_donut.update_layout(margin=dict(t=60, b=20, l=20, r=20), legend_title_text='Risk tier')

driver_bar = px.bar(
    driver_mix,
    x='risk_tier',
    y='share_within_tier',
    color='top_driver_group',
    barmode='group',
    title='Driver Mix by Analytical Risk Tier',
    labels={'share_within_tier': 'Share within tier', 'risk_tier': 'Risk tier', 'top_driver_group': 'Driver group'}
)
driver_bar.update_layout(margin=dict(t=60, b=40, l=40, r=20), yaxis_tickformat='.0%')

threshold_scatter = go.Figure()
threshold_scatter.add_trace(go.Scatter(
    x=threshold_metrics['score_quantile_cutoff'],
    y=threshold_metrics['observed_churn_rate'],
    mode='lines+markers',
    name='Precision / observed churn rate',
    line=dict(color='#c0392b', width=3),
))
threshold_scatter.add_trace(go.Scatter(
    x=threshold_metrics['score_quantile_cutoff'],
    y=threshold_metrics['recall_at_threshold'],
    mode='lines+markers',
    name='Recall',
    line=dict(color='#1f77b4', width=3),
    yaxis='y2',
))
threshold_scatter.update_layout(
    title='Threshold Operating Curve',
    xaxis=dict(title='Score quantile cutoff', tickformat='.0%'),
    yaxis=dict(title='Precision', rangemode='tozero'),
    yaxis2=dict(title='Recall', overlaying='y', side='right', rangemode='tozero'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(t=70, b=40, l=50, r=50),
)

css = """
<style>
:root {
  --vm-primary: #005090;
  --vm-accent: #f39c12;
  --vm-high: #c0392b;
  --vm-medium: #8A6A00;
  --vm-low: #2e8b57;
  --vm-text: #1f2933;
  --vm-muted: #4A5568;
  --vm-border: #d9e2ec;
  --vm-bg: #f7fafc;
  --vm-card: #ffffff;
}
body {font-family: Arial, sans-serif; margin: 32px; color: var(--vm-text); background: var(--vm-bg);}
h1, h2 {color: var(--vm-primary);}
.grid {display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 20px; margin: 20px 0 28px;}
.card {background: var(--vm-card); border: 1px solid var(--vm-border); border-radius: 14px; padding: 18px 20px; box-shadow: 0 8px 24px rgba(15, 23, 42, 0.06);}
.card h3 {margin: 0 0 10px; font-size: 0.95rem; color: var(--vm-muted); text-transform: uppercase; letter-spacing: 0.04em;}
.card .value {font-size: 2rem; font-weight: 700; color: var(--vm-text);}
.card .note {margin-top: 8px; color: var(--vm-muted); font-size: 0.92rem;}
.section {background: var(--vm-card); border: 1px solid var(--vm-border); border-radius: 14px; padding: 22px; margin: 18px 0; box-shadow: 0 8px 24px rgba(15, 23, 42, 0.04);}
table {border-collapse: collapse; width: 100%; font-size: 0.95rem;}
th, td {border: 1px solid var(--vm-border); padding: 10px; text-align: left;}
th {background: #edf2f7;}
.footer {margin-top: 28px; color: var(--vm-muted); font-size: 0.9rem; border-top: 1px solid var(--vm-border); padding-top: 16px;}
</style>
"""

card_specs = [
    ('Precision@Top 10%', precision_top10, 'Top targeting precision on the held-out scored set.'),
    ('High-Risk Churn Rate', high_risk_rate, 'Observed churn rate inside the analytical HIGH tier.'),
    ('ROC AUC', roc_auc_value, 'Overall ranking discrimination under V2C.'),
    ('Average Precision', avg_precision_value, 'Precision-oriented summary under class imbalance.'),
]
card_html = ''.join([
    f"<div class='card'><h3>{label}</h3><div class='value'>{value:.4f}</div><div class='note'>{note}</div></div>"
    for label, value, note in card_specs
])

html_parts = [
    '<html><head><meta charset="utf-8"><title>Churn Monitoring Dashboard</title></head><body>',
    css,
    '<h1>CHURN MONITORING DASHBOARD</h1>',
    f"<div class='grid'>{card_html}</div>",
    "<div class='section'><h2>Risk mix — V2C percentile tiers (analytical)</h2><p><em>Tiers are assigned by score-rank percentile: LOW = bottom 50%, MEDIUM = next 30%, HIGH = top 20%.</em></p>" + risk_donut.to_html(full_html=False, include_plotlyjs='cdn') + risk_mix.to_html(index=False) + '</div>',
    '<div class="section"><h2>Operational tier distribution — n8n V9 fixed probability bands</h2><p><em>The n8n V9 workflow routes customers using fixed probability thresholds applied at execution time (HIGH ≥ 0.75 / MEDIUM 0.45–0.75 / LOW < 0.45). Minor count divergences versus percentile tiers are expected by design.</em></p>' + operational_tier_dist.to_html(index=False) + '</div>',
    '<div class="section"><h2>Model summary metrics</h2>' + summary_metrics.to_html(index=False) + '</div>',
    '<div class="section"><h2>Score-threshold operating table</h2>' + threshold_metrics.to_html(index=False) + threshold_scatter.to_html(full_html=False, include_plotlyjs=False) + '</div>',
    '<div class="section"><h2>Driver mix</h2><p><em>Interactive distribution of dominant driver families within each analytical risk tier.</em></p>' + driver_bar.to_html(full_html=False, include_plotlyjs=False) + driver_mix.to_html(index=False) + '</div>',
    '<div class="section"><h2>Retention action mix</h2>' + action_mix.to_html(index=False) + '</div>',
    '<div class="section"><h2>Notes</h2><ul>'
    '<li>Risk tiers in the analytical tables follow the canonical V2C percentile policy: LOW = bottom 50%, MEDIUM = next 30%, HIGH = top 20% by churn-probability rank.</li>'
    '<li>The operational n8n V9 workflow uses fixed probability bands (HIGH ≥ 0.75 / MEDIUM 0.45–0.75 / LOW &lt; 0.45) applied at execution time.</li>'
    '<li>HIGH risk retains a 15% control group for campaign lift measurement.</li>'
    '<li>SMS remains Phase 3 scope; the current workflow dispatches email and push only.</li>'
    '<li>The target remains highly positive-heavy; calibration should still be interpreted cautiously despite strong ranking metrics.</li>'
    '</ul></div>',
    f"<div class='footer'><strong>Run date:</strong> {run_datetime_label} &nbsp;|&nbsp; <strong>Model version:</strong> {model_version} &nbsp;|&nbsp; <strong>Pipeline tag:</strong> {pipeline_tag}</div>",
    '</body></html>'
]
dashboard_path.write_text('\n'.join(html_parts), encoding='utf-8')
logger.info('Dashboard saved to %s', dashboard_path)
dashboard_path


2026-05-19 16:09:03,313 | INFO | Dashboard saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/reports/churn_monitoring_dashboard_20260519.html


PosixPath('/data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/reports/churn_monitoring_dashboard_20260519.html')

---
## 9.3. NOTEBOOK CLOSURE


The reporting layer now consolidates prediction quality, churn drivers, and campaign readiness into a single lightweight HTML dashboard aligned with the canonical V2C policy.

The dashboard explicitly surfaces the design-level distinction between the analytical tiering criterion (V2C percentile tiers, used throughout NB05–NB09) and the operational tiering criterion (fixed probability bands applied in the n8n V9 workflow: HIGH ≥ 0.75 / MEDIUM 0.45–0.75). Both layers are traceable from a single source column (`churn_probability`) and both target the same high-risk population; the divergence in boundary conditions is intentional and documented here for full auditability.

This closes the initial end-to-end notebook flow from raw data to operational retention outputs. The next evolution (v2) should revisit the eligible population definition as the primary analytical priority, and Phase 3 will extend the n8n Layer 2 pipeline with a hardened customer-facing delivery layer including SMS, production scheduling, channel governance, and a closed-loop measurement framework.